# BindingDB preprocessing — all 5 folds: ProSE/Bepler embeddings + Ames RF labels

This is the updated version of your uploaded `preprocessing_bindind_db.ipynb`.

**Your current layout:** `train_dataset/` has all five original folds, and `test_dataset/` should contain their five matching partitions. Only **d1 (`1042_104`)** currently has ProSE `.h5` embeddings. The notebook now handles **all 5 × (train + test) = 10 embedding files**, before RF labelling or any fine-tuning:

1. Verify all ten original fold folders and their unique-protein TXT/FASTA files, **without changing the protein order**.
2. Report present / missing / invalid `.h5` files. Generate the missing ones using the same ProSE/Bepler `--pool avg` model. `REPLACE_EXISTING_EMBEDDINGS` can also regenerate d1 (backing up the original before replacement).
3. Write/replace the `.h5` **inside its own train/test fold directory**, named `dN_tr_unique_embeddings_bepler.h5` or `dN_te_unique_embeddings_bepler.h5`. Generation is resumable in model-and-input-hashed shards; HDF5 installation is atomic.
4. Validate that every HDF5 has exactly `p1…pN`, 6,165 finite values per embedding, and the number of embeddings equals the corresponding ordered FASTA records.
5. Optionally run RF toxicity labelling and create processed train/val/test TSVs for one fold or all five. Those processed outputs live in versioned directories and are **not silently overwritten**.

**Nothing rebuilds or overwrites your original corrected ligand/protein files, TXT lists, or FASTA files.** Preserve the published train/test protein split; the embedding stage changes HDF5 only.

**Compute:** run ProSE on a scheduled Bhavani allocation (CPU or GPU as configured), *not* the login node. Verify the vendored ProSE script, `bioemb` Python executable, and `prose_mt_3x1024.sav` path before running all ten partitions.

**Scientific note:** the selected RF predicts **Ames mutagenicity**, not all types of toxicity. CTDDG's paper used pkCSM for BindingDB; this RF-based adaptation is a deliberate deviation. Validate against the ChEMBL files actually used in each pretrained model before making a reproduction claim.


In [1]:
from pathlib import Path
import os, csv, json, hashlib, time, random, collections, shutil, tempfile, subprocess, sys
import numpy as np
import pandas as pd
import joblib, h5py
from rdkit import Chem, DataStructs, rdBase, RDLogger
RDLogger.DisableLog('rdApp.error')
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator

REPO_ROOT = Path(os.environ.get('CTDDG_REPO_ROOT', '/home/muhamed/repo/CTDDG'))
BINDINGDB_ROOT = REPO_ROOT / 'datasets/data/bindingdb'
FOLD = 'd1'
FOLD_SUFFIX = {'d1':'4_org_1042_104','d2':'4_org_1000_112','d3':'4_org_1004_122',
               'd4':'4_org_1002_103','d5':'4_org_1036_124'}
# Exactly the atom-type file used for your CURRENT pretraining run, not the older 65-type file.
ATOM_TYPES_FILE = REPO_ROOT / 'REWORKED_CODE/chembl_preprocessed/atom_types.txt'
# RF.ipynb saves these inside rf_outputs, depending on where it was executed.
RF_DIR = REPO_ROOT / 'REWORKED_CODE/Random_Forest/rf_outputs'
RF_FILE = RF_DIR / 'best_ames_rf.joblib'
RF_METADATA_FILE = RF_DIR / 'best_ames_rf_metadata.json'
# Each pretrainer has its OWN ChEMBL input (the CDGCN source is often unlabelled).
PRETRAIN_SOURCES = {
    'ctddg_eq8': {
        'data': REPO_ROOT / 'REWORKED_CODE/chembl_preprocessed/chembl_final.txt',
        'meta': REPO_ROOT / 'REWORKED_CODE/pretraining/pretraining_runs/ctddg_torch_eq8_bs32_v1/run_meta.json',
    },
    'cdgcn_nll': {
        'data': REPO_ROOT / 'REWORKED_CODE/chembl_preprocessed/chembl.txt',
        'meta': REPO_ROOT / 'REWORKED_CODE/pretraining/pretraining_runs/cdgcn_torch_nll_bs32_v1/run_meta.json',
    },
}
# Shards and backups live under processed/. FINAL embeddings live INSIDE every fold folder.
EMBEDDINGS_DIR = REPO_ROOT / 'datasets/processed/prose_embeddings_v1'
PROSE_SCRIPT = REPO_ROOT / 'prose/embed_sequences.py'
PROSE_PYTHON = sys.executable  # set path to `bioemb` Python if current kernel lacks ProSE
PROSE_MODEL = 'prose_mt'       # the published ProSE model (saved_models/ must be installed)
PROSE_DEVICE = -1            # CPU; use 0 only inside a permitted GPU allocation
EMBED_SHARD_SIZE = 64
# Run this stage BEFORE RF labelling. It covers ALL five folds, train + test.
RUN_ALL_FOLD_EMBEDDINGS = True
GENERATE_MISSING_EMBEDDINGS = True
# True also regenerates existing d1 embeddings; originals are backed up before replacement.
# False saves work by retaining already valid HDF5s.
REPLACE_EXISTING_EMBEDDINGS = False
# The model file, when provided, must be the exact one used for 6165-d embeddings.
PROSE_SAVED_MODEL = REPO_ROOT / 'prose/saved_models/prose_mt_3x1024.sav'

OUTPUT_DIR = REPO_ROOT / f'datasets/processed/ctddg_bindingdb_rf_{FOLD}_v2'
RUN_PREPROCESSING = False        # True writes a processed dataset only for FOLD
RUN_PREPROCESSING_ALL_FOLDS = True  # True writes separate processed datasets for d1..d5
PROCESSED_DATASET_VERSION = 'v2'  # change to v3 if the output directory already exists
CHECK_PRETRAIN_LEAKAGE = True # Direct set-intersection against the EXACT stage-1 molecules
BUILD_ALL_FOLD_EXCLUSION = True # canonical ligands from all five folds; relatively CPU intensive
REQUIRE_EMBEDDINGS = True   # Fail rather than falsely assume protein vectors exist
N_C = 6165
MAX_ATOMS, MAX_STEPS = 100, 120
VAL_PROTEIN_FRACTION, SPLIT_SEED = 0.05, 42
PREDICT_BATCH = 8192
# Output folder is immutable once created: change version/suffix for a new experiment.
print('RDKit:',rdBase.rdkitVersion)
for k,v in {'raw_bindingdb':BINDINGDB_ROOT,'atom_vocab':ATOM_TYPES_FILE,
            'RF':RF_FILE,'RF_metadata':RF_METADATA_FILE,'pretrain_sources': PRETRAIN_SOURCES,
            'output':OUTPUT_DIR}.items(): print(f'{k:15s}',v,'exists:',(v.exists() if isinstance(v,Path) else {x:{y:str(z) for y,z in r.items()} for x,r in v.items()}))
assert FOLD in FOLD_SUFFIX


RDKit: 2026.03.6
raw_bindingdb   /home/muhamed/repo/CTDDG/datasets/data/bindingdb exists: True
atom_vocab      /home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/atom_types.txt exists: True
RF              /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/rf_outputs/best_ames_rf.joblib exists: True
RF_metadata     /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/rf_outputs/best_ames_rf_metadata.json exists: True
pretrain_sources {'ctddg_eq8': {'data': PosixPath('/home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/chembl_final.txt'), 'meta': PosixPath('/home/muhamed/repo/CTDDG/REWORKED_CODE/pretraining/pretraining_runs/ctddg_torch_eq8_bs32_v1/run_meta.json')}, 'cdgcn_nll': {'data': PosixPath('/home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/chembl.txt'), 'meta': PosixPath('/home/muhamed/repo/CTDDG/REWORKED_CODE/pretraining/pretraining_runs/cdgcn_torch_nll_bs32_v1/run_meta.json')}} exists: {'ctddg_eq8': {'data': '/home/muhamed/repo/CTDDG/REWORKED_CODE/chem

## 1. Strict file discovery and alignment

Ligand and protein files are **paired line-for-line**. Their lines may contain *space-separated characters*. We reassemble each **whole line** with `''.join(line.split())` before parsing it as a molecule/protein. **Never use `line.split()[0]` for BindingDB ligands.** The per-fold protein lists and FASTA are *not* sorted/reconstructed: the original order determines `p1`, `p2`, etc. in ProSE HDF5.


In [2]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()

def fold_dir(fold,part):
    return BINDINGDB_ROOT / f'{part}_dataset' / f'{part}_{FOLD_SUFFIX[fold]}'

def find_pair_files(fold,part):
    d=fold_dir(fold,part)
    files={}
    for kind in ('ligands','proteins'):
        # Both naming variants appear in the old preprocessors.
        options=[d/f'{kind}_{part}_4_org_corrected', d/f'{kind}_{FOLD_SUFFIX[fold]}_corrected']
        found=[x for x in options if x.is_file()]
        if not found:
            found=sorted(x for x in d.glob(f'{kind}*_corrected') if x.is_file())
        if len(found)!=1:
            raise RuntimeError(f'{kind} in {d}: expected one matching file, found {found}')
        files[kind]=found[0]
    return files

def protein_paths(fold,part):
    tag='tr' if part=='train' else 'te'; d=fold_dir(fold,part)
    return {'list':d/f'{fold}_{tag}_unique_proteins.txt',
            'fasta':d/f'{fold}_{tag}_unique_proteins_bepler.fa',
            'h5': d/f'{fold}_{tag}_unique_embeddings_bepler.h5'}

def spaced_lines(path):
    with open(path) as f:
        for line_no,line in enumerate(f,1):
            v=''.join(line.split())
            if not v: raise ValueError(f'Blank line at {path}:{line_no}; pair alignment at risk')
            yield v

def read_fasta(path):
    records=[]; name=None; seq=[]
    with open(path) as f:
        for line in f:
            line=line.strip()
            if line.startswith('>'):
                if name is not None: records.append((name,''.join(seq).upper()))
                name=line[1:].split()[0];seq=[]
            elif line: seq.append(''.join(line.split()))
    if name is not None: records.append((name,''.join(seq).upper()))
    return records

def verify_protein_order(fold,part,check_embeddings=True):
    files=protein_paths(fold,part)
    for key in ('list','fasta'):
        if not files[key].is_file():raise FileNotFoundError(files[key])
    sequences=[s.upper() for s in spaced_lines(files['list'])]
    if len(sequences)!=len(set(sequences)):
        raise ValueError(f'{files["list"]}: duplicated sequences make mapping ambiguous')
    fasta=read_fasta(files['fasta'])
    if len(fasta)!=len(sequences):raise ValueError(f'FASTA/list count mismatch for {fold}/{part}')
    for i,((name,seq),actual) in enumerate(zip(fasta,sequences),1):
        if name!=f'p{i}' or seq!=actual:
            raise ValueError(f'{fold}/{part}: FASTA/list disagree at index {i}, id {name!r}')
    if REQUIRE_EMBEDDINGS and check_embeddings:
        ensure_prose_embeddings(fold, part, sequences, files)
        with h5py.File(files['h5'],'r') as h:
            if set(h.keys())!={f'p{i}' for i in range(1,len(sequences)+1)}:
                raise ValueError('HDF5 keys must be exactly p1..pN (same FASTA order)')
            for i in range(1,len(sequences)+1):
                x=np.asarray(h[f'p{i}'])
                if x.shape!=(N_C,) or not np.isfinite(x).all():
                    raise ValueError(f'{files["h5"]}: p{i} shape {x.shape} or non-finite; expected ({N_C},)')
    return sequences, files

def read_paired(fold,part):
    f=find_pair_files(fold,part)
    seen=set(); records=[]
    import itertools
    from itertools import zip_longest
    with open(f['ligands']) as ls,open(f['proteins']) as ps:
        for i,pair in enumerate(zip_longest(ls,ps),1):
            ligand,protein=pair
            if ligand is None or protein is None:
                raise ValueError(f'Ligand/protein row counts mismatch: {f} at line {i}')
            smi,seq=''.join(ligand.split()),''.join(protein.split()).upper()
            if not smi or not seq:raise ValueError(f'Empty ligand/protein: {f} at line {i}')
            key=(smi,seq)
            if key not in seen:
                records.append((smi,seq,part,i));seen.add(key)
    return records, {k:str(v) for k,v in f.items()}


## 2. Generate all ten ProSE/Bepler `.h5` files (before preprocessing)

The existing TXT and FASTA are **authoritative** for the order of the protein IDs. They are validated but **never reconstructed or re-sorted**. A protein at FASTA position `i` must be stored in HDF5 under `p{i}`.

The official ProSE embedding script is invoked with `--pool avg --model prose_mt`, yielding the **6,165-dimensional** Bepler/ProSE vectors expected by the conditional generator. Its model checkpoint must be available at `PROSE_SAVED_MODEL`, and `PROSE_PYTHON` must point to the Python environment with ProSE dependencies.

**Settings in the configuration cell:**

- `RUN_ALL_FOLD_EMBEDDINGS=True` runs this stage for every fold's training and test partition. If disabled, it only reports statuses.
- `GENERATE_MISSING_EMBEDDINGS=True` allows missing vectors to be computed. Disable to do inventory-only checks.
- `REPLACE_EXISTING_EMBEDDINGS=False` **reuses** any existing valid HDF5; change to `True` to regenerate d1 too, with a timestamped backup of each replaced HDF5.
- `PROSE_DEVICE=-1` uses CPU. Set `0` for the allocated GPU only when running on a compute node.

Per-shard outputs are cached outside the original dataset and can be reused after interruption. Input FASTA content, pooling method, model weights, and embedding-script hash are part of the shard cache key.


In [3]:
def _check_hdf5(h5path, sequences):
    h5path = Path(h5path)
    if not h5path.is_file():
        return False
    with h5py.File(h5path, 'r') as h:
        expected = {f'p{i}' for i in range(1, len(sequences) + 1)}
        if set(h.keys()) != expected:
            raise ValueError(f'HDF5 IDs incorrect: {h5path} (expected p1..p{len(sequences)})')
        for i in range(1, len(sequences) + 1):
            vec = np.asarray(h[f'p{i}'], dtype=np.float32)
            if vec.shape != (N_C,) or not np.isfinite(vec).all():
                raise ValueError(f'Embedding {h5path}:p{i} has shape {vec.shape} or nonfinite values')
    return True


def ensure_prose_embeddings(fold, part, sequences, paths, *, force=False):
    """Create missing (or explicitly replace) HDF5, using ordered FASTA and resumable shards.

    Only the .h5 is written under the original fold; input TXT/FASTA/pairs are untouched.
    """
    outfile = Path(paths['h5'])
    if outfile.is_file() and not force:
        _check_hdf5(outfile, sequences)
        return outfile
    if not GENERATE_MISSING_EMBEDDINGS:
        raise FileNotFoundError(f'Missing/overwrite requested: {outfile}. Set GENERATE_MISSING_EMBEDDINGS=True.')
    if not PROSE_SCRIPT.is_file():
        raise FileNotFoundError(f'Vendored ProSE script not found: {PROSE_SCRIPT}')
    if not PROSE_SAVED_MODEL.is_file():
        raise FileNotFoundError(f'Missing ProSE model: {PROSE_SAVED_MODEL}')
    if EMBED_SHARD_SIZE < 1:
        raise ValueError('EMBED_SHARD_SIZE must be positive')
    if not sequences:
        raise ValueError(f'No proteins for {fold}/{part}')

    outfile.parent.mkdir(parents=True, exist_ok=True)
    shard_dir = EMBEDDINGS_DIR / 'shards' / f'{fold}_{part}'
    shard_dir.mkdir(parents=True, exist_ok=True)
    model_sha = sha256(PROSE_SAVED_MODEL)
    script_sha = sha256(PROSE_SCRIPT)
    all_shards = []
    for start in range(0, len(sequences), EMBED_SHARD_SIZE):
        end = min(start + EMBED_SHARD_SIZE, len(sequences))
        content = ''.join(f'>p{i+1}\n{sequences[i]}\n' for i in range(start, end))
        token = f'{model_sha}|{script_sha}|avg|{N_C}|{content}'
        identity = hashlib.sha256(token.encode()).hexdigest()[:16]
        prefix = f'{start:05d}_{end:05d}_{identity}'
        fa = shard_dir / f'{prefix}.fa'
        target = shard_dir / f'{prefix}.h5'
        expected = {f'p{i+1}' for i in range(start, end)}
        if target.exists():
            try:
                with h5py.File(target, 'r') as check:
                    if set(check.keys()) != expected:
                        raise ValueError('Wrong IDs')
                    for key in expected:
                        arr = np.asarray(check[key])
                        if arr.shape != (N_C,) or not np.isfinite(arr).all():
                            raise ValueError('Bad vector')
            except (OSError, ValueError) as exc:
                print(f'Invalid cached shard ({exc}), regenerating: {target}')
                target.unlink()
        if not target.exists():
            fa.write_text(content)
            tmp = shard_dir / f'{prefix}.tmp.{os.getpid()}.h5'
            command = [str(PROSE_PYTHON), str(PROSE_SCRIPT), '-d', str(PROSE_DEVICE),
                       '--pool', 'avg', '--model', str(PROSE_MODEL), '-o', str(tmp), str(fa)]
            print(f'[{fold}/{part}] Embedding shard {start+1}..{end} / {len(sequences)}', flush=True)
            try:
                subprocess.run(command, cwd=str(PROSE_SCRIPT.parent), check=True)
                with h5py.File(tmp, 'r') as check:
                    if set(check.keys()) != expected:
                        raise ValueError(f'ProSE wrote incorrect IDs in {tmp}')
                    for key in expected:
                        arr = np.asarray(check[key])
                        if arr.shape != (N_C,) or not np.isfinite(arr).all():
                            raise ValueError(f'Bad ProSE vector {key}: {arr.shape}')
                os.replace(tmp, target)
            finally:
                if tmp.exists():
                    tmp.unlink()
        all_shards.append(target)

    tmp_out = outfile.with_name(outfile.name + f'.tmp.{os.getpid()}')
    try:
        with h5py.File(tmp_out, 'w') as merged:
            for shard in all_shards:
                with h5py.File(shard, 'r') as part_h5:
                    for key in part_h5:
                        merged.create_dataset(key, data=np.asarray(part_h5[key], dtype=np.float32))
            merged.attrs['prose_model_sha256'] = model_sha
            merged.attrs['embed_script_sha256'] = script_sha
            merged.attrs['fasta_sha256'] = sha256(paths['fasta'])
            merged.attrs['sequence_list_sha256'] = sha256(paths['list'])
            merged.attrs['pool'] = 'avg'
            merged.attrs['n_c'] = N_C
        _check_hdf5(tmp_out, sequences)

        if outfile.exists():
            if not force:
                raise FileExistsError(f'Unexpected existing HDF5: {outfile}')
            backup_dir = EMBEDDINGS_DIR / 'backups' / f'{fold}_{part}'
            backup_dir.mkdir(parents=True, exist_ok=True)
            stamp = time.strftime('%Y%m%d_%H%M%S')
            backup = backup_dir / f'{outfile.name}.{stamp}.{os.getpid()}.bak'
            shutil.copy2(outfile, backup)
            if sha256(backup) != sha256(outfile):
                raise RuntimeError(f'Backup verification failed; refusing to replace {outfile}')
            print(f'[{fold}/{part}] Original backed up to {backup}')
        os.replace(tmp_out, outfile)  # atomic final installation on same filesystem
    finally:
        if tmp_out.exists():
            tmp_out.unlink()
    print(f'[{fold}/{part}] Verified {len(sequences)} × {N_C} and saved: {outfile}')
    return outfile


def preflight_all_embedding_folds():
    """Validate every fold's TXT/FASTA alignment before starting ANY embedding jobs."""
    jobs = []
    errors = []
    for fold in FOLD_SUFFIX:  # expected d1..d5, never fabricate missing folds
        for part in ('train', 'test'):
            try:
                d = fold_dir(fold, part)
                if not d.is_dir():
                    raise FileNotFoundError(d)
                seqs, paths = verify_protein_order(fold, part, check_embeddings=False)
                pair_files = find_pair_files(fold, part)
                h5status = 'MISSING'
                if paths['h5'].exists():
                    try:
                        _check_hdf5(paths['h5'], seqs)
                        h5status = 'VALID'
                    except (OSError, ValueError) as ex:
                        h5status = f'INVALID ({ex})'
                        if not (GENERATE_MISSING_EMBEDDINGS and REPLACE_EXISTING_EMBEDDINGS):
                            errors.append(f'{fold}/{part}: invalid HDF5; enable replacement: {ex}')
                print(f'{fold:2s} {part:5s}: {len(seqs):4d} proteins | H5 {h5status} | {paths["h5"]}')
                jobs.append((fold, part, seqs, paths, pair_files, h5status))
            except Exception as ex:
                errors.append(f'{fold}/{part}: {type(ex).__name__}: {ex}')
    if errors:
        raise RuntimeError('Missing/invalid BindingDB inputs (no embeddings run):\n'+'\n'.join(errors))
    if len(jobs) != 10:
        raise RuntimeError(f'Expected 10 train/test partitions, found {len(jobs)}')
    return jobs


def run_all_fold_embeddings():
    """Preflight all ten partitions, then write missing or replace existing .h5 files."""
    jobs = preflight_all_embedding_folds()
    needs = [j for j in jobs if REPLACE_EXISTING_EMBEDDINGS or j[-1] != 'VALID']
    if needs and not GENERATE_MISSING_EMBEDDINGS:
        raise RuntimeError(f'{len(needs)} HDF5 outputs need generation. Enable GENERATE_MISSING_EMBEDDINGS.')
    if needs:
        if not PROSE_SCRIPT.is_file() or not PROSE_SAVED_MODEL.is_file():
            raise FileNotFoundError(f'Check ProSE script/model paths: {PROSE_SCRIPT}, {PROSE_SAVED_MODEL}')
        print(f'\nGenerating {len(needs)} of 10 HDF5 files; replacement enabled: {REPLACE_EXISTING_EMBEDDINGS}\n')
    status = []
    for fold, part, seqs, paths, pair_files, before in jobs:
        force = bool(REPLACE_EXISTING_EMBEDDINGS and paths['h5'].exists())
        ensure_prose_embeddings(fold, part, seqs, paths, force=force)
        _check_hdf5(paths['h5'], seqs)
        status.append({'fold': fold, 'part': part, 'proteins': len(seqs),
                       'state': 'REGENERATED' if force else ('GENERATED' if before!='VALID' else 'REUSED'),
                       'h5': str(paths['h5']), 'h5_sha256': sha256(paths['h5'])})
    assert len(status) == 10
    print('\nALL 5 FOLDS VERIFIED — 10 / 10 embedding files, each with 6165-d vectors')
    for s in status:
        print(f'{s["fold"]} {s["part"]:5s} {s["state"]:11s} {s["proteins"]} × {N_C}')
    return status


print('ProSE available:', PROSE_SCRIPT.is_file(), 'saved model:', PROSE_SAVED_MODEL.is_file())
if RUN_ALL_FOLD_EMBEDDINGS:
    embedding_report = run_all_fold_embeddings()
else:
    print('Inventory only — set RUN_ALL_FOLD_EMBEDDINGS=True to generate embeddings for all five folds.')
    embedding_inventory = preflight_all_embedding_folds()


ProSE available: True saved model: True
d1 train: 1042 proteins | H5 VALID | /home/muhamed/repo/CTDDG/datasets/data/bindingdb/train_dataset/train_4_org_1042_104/d1_tr_unique_embeddings_bepler.h5
d1 test :  104 proteins | H5 VALID | /home/muhamed/repo/CTDDG/datasets/data/bindingdb/test_dataset/test_4_org_1042_104/d1_te_unique_embeddings_bepler.h5
d2 train: 1000 proteins | H5 VALID | /home/muhamed/repo/CTDDG/datasets/data/bindingdb/train_dataset/train_4_org_1000_112/d2_tr_unique_embeddings_bepler.h5
d2 test :  112 proteins | H5 VALID | /home/muhamed/repo/CTDDG/datasets/data/bindingdb/test_dataset/test_4_org_1000_112/d2_te_unique_embeddings_bepler.h5
d3 train: 1004 proteins | H5 VALID | /home/muhamed/repo/CTDDG/datasets/data/bindingdb/train_dataset/train_4_org_1004_122/d3_tr_unique_embeddings_bepler.h5
d3 test :  122 proteins | H5 VALID | /home/muhamed/repo/CTDDG/datasets/data/bindingdb/test_dataset/test_4_org_1004_122/d3_te_unique_embeddings_bepler.h5
d4 train: 1002 proteins | H5 VALID |

## 2. Molecule cleaning and checkpoint vocabulary

The cleaning policy matches the **heavy-atom principal-fragment** choice in `RF.ipynb`. RDKit canonicalization happens before RF labelling and ligand equality checks. Disconnected molecules do not enter graph traversal. `atom_types.txt` is an **ordered checkpoint contract**: OOV ligands are rejected and never added to the vocabulary.


In [4]:
BONDS={Chem.BondType.AROMATIC,Chem.BondType.SINGLE,Chem.BondType.DOUBLE,Chem.BondType.TRIPLE}

def load_vocab():
    if not ATOM_TYPES_FILE.is_file():raise FileNotFoundError(ATOM_TYPES_FILE)
    rows=[]
    for ln,line in enumerate(ATOM_TYPES_FILE.read_text().splitlines(),1):
        if not line.strip():continue
        x=line.strip().split(',')
        if len(x)!=3:raise ValueError(f'Bad atom type at line {ln}')
        rows.append((x[0],int(x[1]),int(x[2])))
    if len(rows)!=len(set(rows)) or not rows:raise ValueError('Duplicate/empty atom vocabulary')
    return set(rows)

def clean_smiles(smi,allowed_atoms=None):
    # Invalid BindingDB records must be reported and skipped, not crash a large
    # CPU preprocessing run after hundreds of thousands of valid records.
    try:
        mol=Chem.MolFromSmiles(smi)
        if mol is None or not mol.GetNumAtoms():return None,'invalid_smiles'
        parts=Chem.GetMolFrags(mol,asMols=True,sanitizeFrags=True)
        if len(parts)>1:
            mol=max(parts,key=lambda x:(x.GetNumHeavyAtoms(),x.GetNumAtoms(),Chem.MolToSmiles(x)))
        if mol.GetNumBonds()==0:return None,'zero_bonds'
        if mol.GetNumAtoms()>MAX_ATOMS:return None,'too_many_atoms'
        if mol.GetNumBonds()+1>MAX_STEPS:return None,'too_many_actions'
        if any(b.GetBondType() not in BONDS for b in mol.GetBonds()):return None,'unsupported_bond'
        if allowed_atoms is not None:
            if any((a.GetSymbol(),a.GetFormalCharge(),a.GetNumExplicitHs()) not in allowed_atoms
                   for a in mol.GetAtoms()):return None,'out_of_vocab'
        return Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True),None
    except Exception:
        return None,'rdkit_exception'

# Two important parser / chemistry regression tests
assert ''.join('C C ( = O ) O'.split())=='CC(=O)O'
assert clean_smiles('CC(=O)Oc1ccccc1C(=O)O.Cl')[0] == Chem.MolToSmiles(Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O'))
print('PASS: whitespace reconstruction and principal-fragment tests')


PASS: whitespace reconstruction and principal-fragment tests


## 3. Label with **exact selected RF weights**

Reads `best_ames_rf.joblib` plus its `best_ames_rf_metadata.json` from **your new RF notebook**, rather than the older `ames_random_forest.pkl` referred to in historical fine-tuning code. The metadata fixes the feature type/radius/width. The RF is never retrained automatically. Only load joblib files you trust.


In [5]:
def load_selected_rf():
    if not RF_FILE.is_file() or not RF_METADATA_FILE.is_file():
        raise FileNotFoundError(f'Need BOTH {RF_FILE} and {RF_METADATA_FILE}. Check RF.ipynb OUTPUT_DIR.')
    rf=joblib.load(RF_FILE) # trusted own file, pickle-based
    meta=json.loads(RF_METADATA_FILE.read_text())
    cfg=meta.get('selected_fingerprint')
    if not isinstance(cfg,dict):raise ValueError('RF metadata missing selected_fingerprint')
    kind=cfg.get('kind')
    if kind not in ('morgan','maccs'):raise ValueError(f'Unsupported RF fingerprint kind: {kind}')
    n_bits=int(cfg.get('n_bits',167 if kind=='maccs' else 2048))
    radius=int(cfg.get('radius',2))
    if kind=='maccs' and n_bits!=167:raise ValueError('RDKit MACCS feature vector must have 167 bits')
    if getattr(rf,'n_features_in_',None)!=n_bits:
        raise ValueError(f'RF expects {getattr(rf,"n_features_in_",None)} features but recipe says {n_bits}')
    if set(map(int,rf.classes_))!={0,1}:raise ValueError(f'RF class encoding {rf.classes_} is not 0/1')
    fp={'kind':kind,'n_bits':n_bits,'radius':radius}
    print('Selected RF:',meta.get('selected_model'), '| features:',fp,'| weights SHA:',sha256(RF_FILE)[:16])
    return rf,fp,meta

def fingerprint_matrix(smiles,recipe):
    n_bits=recipe['n_bits'];a=np.zeros((len(smiles),n_bits),dtype=np.uint8)
    gen=(rdFingerprintGenerator.GetMorganGenerator(radius=recipe['radius'],fpSize=n_bits)
         if recipe['kind']=='morgan' else None)
    for i,s in enumerate(smiles):
        mol=Chem.MolFromSmiles(s)
        if mol is None:raise ValueError(f'Previously cleaned SMILES invalid: {s}')
        bitvect=gen.GetFingerprint(mol) if gen is not None else MACCSkeys.GenMACCSKeys(mol)
        DataStructs.ConvertToNumpyArray(bitvect,a[i])
    return a

def label_unique(unique_smiles):
    rf,recipe,meta=load_selected_rf()
    labels={}
    smi=sorted(set(unique_smiles))
    for off in range(0,len(smi),PREDICT_BATCH):
        chunk=smi[off:off+PREDICT_BATCH]
        pred=rf.predict(fingerprint_matrix(chunk,recipe))
        if not set(np.unique(pred)).issubset({0,1}):raise ValueError('RF output outside 0/1')
        labels.update((s,int(y)) for s,y in zip(chunk,pred))
        if (off//PREDICT_BATCH)%20==0:print(f'  RF: {min(off+len(chunk),len(smi)):,}/{len(smi):,} unique ligands')
    return labels,recipe


## 4. Assemble fold, make protein-level validation, verify leakage

The paired raw files can include proteins outside their designated folds. We route records according to the authoritative train/test **sequence lists**; we pool both raw pair files to retain all associations for their corresponding proteins. Duplicate (ligand, protein) pairs are removed, but the same ligand may correctly bind more than one protein. We do not use any held-out protein for fine-tuning or model selection.


In [6]:
def assemble_fold(fold=FOLD):
    seq_train,files_train=verify_protein_order(fold,'train')
    seq_test,files_test=verify_protein_order(fold,'test')
    if set(seq_train)&set(seq_test):raise ValueError('Train/test protein sequence overlap')
    maps={'train':{s:i for i,s in enumerate(seq_train)},'test':{s:i for i,s in enumerate(seq_test)}}
    gathered=[];raw_files={}
    for part in ('train','test'):
        pairs,paths=read_paired(fold,part)
        raw_files.update({f'{part}_{kind}':p for kind,p in paths.items()})
        gathered+=pairs
        print(part,'raw unique pairs:',len(pairs))
    seen=set();rows={'train':[],'test':[]};stats=collections.Counter()
    allowed=load_vocab()
    for raw,seq,source,line_no in gathered:
        partitions=[part for part in ('train','test') if seq in maps[part]]
        if not partitions:
            stats['outside_fold_proteins']+=1;continue
        if len(partitions)!=1:raise ValueError('Same protein in two partitions')
        part=partitions[0]
        clean,why=clean_smiles(raw,allowed)
        if why: stats[f'{part}_{why}']+=1;continue
        key=(part,seq,clean)
        if key in seen:stats['duplicate_pair']+=1;continue
        seen.add(key)
        rows[part].append({'smiles':clean,'protein_index':maps[part][seq],
                           'protein_id':f'p{maps[part][seq]+1}',
                           'source_partition':source,'source_line':line_no})
    for part, seqs in [('train', seq_train), ('test', seq_test)]:
        have = {r['protein_index'] for r in rows[part]}
        missing = set(range(len(seqs))) - have

        if not have:
            raise RuntimeError(
                f"{fold}/{part}: no usable protein-ligand pairs"
            )

        if missing:
            print(
                f"[WARNING] {fold}/{part}: "
                f"{len(missing)} proteins excluded "
                "because they have no usable ligands."
            )
            print(
                "Protein IDs:",
                [f"p{i+1}" for i in sorted(missing)]
            )

        print(
            part,
            len(rows[part]), "pairs;",
            len(have), "usable proteins"
        )
    rng = np.random.default_rng(SPLIT_SEED)

    active_train = sorted({
        r['protein_index'] for r in rows['train']
    })

    if len(active_train) < 2:
        raise RuntimeError(
            "Need at least two usable training proteins"
        )

    n_val = max(
        1,
        round(VAL_PROTEIN_FRACTION * len(active_train))
    )
    n_val = min(n_val, len(active_train) - 1)

    val_indices = set(
        map(int, rng.permutation(active_train)[:n_val])
    )
    train=[r for r in rows['train'] if r['protein_index'] not in val_indices]
    val=[r for r in rows['train'] if r['protein_index'] in val_indices]
    test=rows['test']
    if not train or not val or not test:raise RuntimeError('Empty train, val or test split')
    for group in (train,val,test):
        if any(r['protein_id']!=f'p{r["protein_index"]+1}' for r in group):raise AssertionError('Protein index error')
    if {r['protein_index'] for r in train}&{r['protein_index'] for r in val}:
        raise AssertionError('Train/val share proteins')
    overlap=len({r['smiles'] for r in train}|{r['smiles'] for r in val})
    print('Validation protein rows:',len(val_indices),'ligand/protein pool unique ligands:',overlap)
    return {'train':train,'val':val,'test':test}, {'train':files_train,'test':files_test}, raw_files, dict(stats), sorted(val_indices)

def all_fold_ligands():
    result=set();counts=collections.Counter();paths={}
    for fold in FOLD_SUFFIX:
        for part in ('train','test'):
            p=find_pair_files(fold,part)['ligands'];paths[f'{fold}_{part}']=str(p)
            for raw in spaced_lines(p):
                smi,err=clean_smiles(raw,allowed_atoms=None)
                if smi: result.add(smi)
                else:counts[err]+=1
    print('All-fold canonical ligands:',len(result),'rejected:',dict(counts))
    return result,paths

def pretrain_overlap(exclusion, variant):
    """Check EXACT stage-1 dataset, cross-check its SHA against stage-1 run_meta."""
    info=PRETRAIN_SOURCES[variant]
    data,meta=Path(info['data']),Path(info['meta'])
    if not data.is_file() or not meta.is_file():
        return {'status':'UNVERIFIED','reason':f'No exact input/metadata for {variant}: {data} / {meta}'}
    stage1=json.loads(meta.read_text())
    data_hash=sha256(data)
    if stage1.get('chembl_sha256')!=data_hash:
        return {'status':'FAILED','reason':'Stage-1 metadata input SHA differs from the file checked',
                'pretrain_file':str(data),'pretrain_sha256':data_hash,
                'stage1_expected_sha256':stage1.get('chembl_sha256')}
    n=0;bad=0;examples=[];seen=set();labels=collections.Counter()
    with data.open() as reader:
        for line_no,line in enumerate(reader,1):
            line=line.strip()
            if not line:continue
            if variant=='ctddg_eq8':
                fields=line.rsplit(maxsplit=1)
                if len(fields)!=2 or fields[-1] not in ('0','1'):
                    bad+=1;continue
                smi,label=fields;labels[label]+=1
            else:
                # no-toxicity pretrainer uses chembl.txt: SMILES alone (fallback has a dummy label)
                smi=line.split()[0]
            normalized,err=clean_smiles(smi,allowed_atoms=None)
            if err:bad+=1;continue
            if normalized in exclusion and normalized not in seen:
                n+=1;seen.add(normalized)
                if len(examples)<10:examples.append(normalized)
    all_labels_valid = (variant!='ctddg_eq8' or labels['0']>0 and labels['1']>0)
    return {'status':'verified_zero' if n==0 and bad==0 and all_labels_valid else 'FAILED',
            'overlapping_unique_ligands':n,'malformed_or_excluded_pretrain_rows':bad,
            'toxic_label_counts':dict(labels),'label_classes_ok':bool(all_labels_valid),
            'examples':examples,'pretrain_file':str(data),'pretrain_sha256':data_hash,
            'stage1_meta_file':str(meta),'stage1_meta_sha256':sha256(meta)}


## 5. Write versioned artifacts atomically

The output is **new** (never write over a previous training dataset). The manifest contains hashes for the exact raw paired data, FASTA, HDF5 embeddings, RF weights/metadata, atom vocabulary and produced TSVs, so fine-tuning can refuse changed dependencies. `pretrain_leakage.status` is `verified_zero` **only when a direct full-molecule comparison passes**. All-fold filtering is recommended for paper-quality claims.


In [ ]:
def prepare(fold=FOLD):
    output_dir = REPO_ROOT / f'datasets/processed/ctddg_bindingdb_rf_{fold}_{PROCESSED_DATASET_VERSION}'
    if output_dir.exists():
        raise FileExistsError(f'{output_dir} already exists. Use a NEW versioned output directory; do not silently overwrite a run dataset.')
    if not BUILD_ALL_FOLD_EXCLUSION and CHECK_PRETRAIN_LEAKAGE:
        raise ValueError('A claimed all-fold leakage check requires BUILD_ALL_FOLD_EXCLUSION=True')
    rows,split_files,raw_files,filters,val_indices=assemble_fold(fold)
    unique={r['smiles'] for group in rows.values() for r in group}
    labels,recipe=label_unique(unique)
    for group in rows.values():
        for row in group:row['ames']=labels[row['smiles']]
    exclusion,extra_paths=all_fold_ligands() if BUILD_ALL_FOLD_EXCLUSION else (None,{})
    leakage=({variant:pretrain_overlap(exclusion,variant) for variant in PRETRAIN_SOURCES}
             if CHECK_PRETRAIN_LEAKAGE else
             {variant:{'status':'UNVERIFIED','reason':'direct leakage check disabled'}
              for variant in PRETRAIN_SOURCES})
    for variant,item in leakage.items():
        if item['status']!='verified_zero':
            print(f'[BLOCKER] {variant} pretraining ligand exclusion not verified:',item)
    output_dir.parent.mkdir(parents=True,exist_ok=True)
    tmp=Path(tempfile.mkdtemp(prefix='.bindingdb_tmp_',dir=output_dir.parent))
    try:
        for split in ('train','val','test'):
            pd.DataFrame(rows[split],columns=['smiles','protein_index','protein_id','ames','source_partition','source_line']).to_csv(
                tmp/f'{split}.tsv',sep='	',index=False)
        if exclusion is not None:
            with (tmp/'all_bindingdb_ligands.smi').open('w') as f:
                for smi in sorted(exclusion):f.write(smi+'\n')
        dependencies=[ATOM_TYPES_FILE,RF_FILE,RF_METADATA_FILE]
        for files in split_files.values():dependencies+=list(files.values())
        dependencies+=[Path(x) for x in raw_files.values()]
        dependencies+=[Path(x) for x in extra_paths.values()]
        for info in PRETRAIN_SOURCES.values():
            dependencies += [Path(x) for x in info.values() if Path(x).is_file()]
        files_hash={str(path):sha256(path) for path in sorted(set(dependencies))}
        counts={split:{'pairs':len(rows[split]),'unique_proteins':len({r['protein_index'] for r in rows[split]}),
                       'unique_ligands':len({r['smiles'] for r in rows[split]}),
                       'ames_positive':sum(r['ames'] for r in rows[split])} for split in rows}
        manifest={'schema_version':2,'fold':fold,'source':'Grechishnikova BindingDB folds',
                  'label_meaning':'RF-predicted Ames positive (1) vs negative (0), not measured',
                  'paper_deviation':'BindingDB Ames labels are selected RF predictions; CTDDG paper used pkCSM',
                  'rf_model':str(RF_FILE),'rf_recipe':recipe,'atom_types_file':str(ATOM_TYPES_FILE),
                  'protein_paths':{split:{k:str(v) for k,v in files.items()} for split,files in split_files.items()},
                  'protein_rows_val':val_indices,'val_fraction':VAL_PROTEIN_FRACTION,'split_seed':SPLIT_SEED,
                  'max_atoms':MAX_ATOMS,'max_steps':MAX_STEPS,'fragment_policy':'max heavy atoms then atom count then canonical smiles',
                  'pool_pair_files':True,'pair_dedup':'protein + canonical ligand',
                  'stats':counts,'filter_counts':filters,'pretrain_leakage':leakage,'pretrain_sources':{k:{kk:str(vv) for kk,vv in v.items()} for k,v in PRETRAIN_SOURCES.items()},
                  'dependencies_sha256':files_hash,
                  'output_sha256':{f'{s}.tsv':sha256(tmp/f'{s}.tsv') for s in rows}}
        if exclusion is not None:manifest['output_sha256']['all_bindingdb_ligands.smi']=sha256(tmp/'all_bindingdb_ligands.smi')
        (tmp/'dataset_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
        os.rename(tmp,output_dir)
    except Exception:
        shutil.rmtree(tmp,ignore_errors=True);raise
    print('\nDATASET READY:',output_dir)
    print(json.dumps({'stats':counts,'leakage':leakage},indent=2))
    print('Do not start fine-tuning if leakage is FAILED/UNVERIFIED or RF provenance is unresolved.')
    return manifest

if RUN_PREPROCESSING_ALL_FOLDS:
    if not RUN_ALL_FOLD_EMBEDDINGS:
        raise RuntimeError('First run the all-fold embedding stage and verify its outputs.')
    manifests = {fold: prepare(fold) for fold in FOLD_SUFFIX}
elif RUN_PREPROCESSING:
    manifest = prepare(FOLD)
else:
    print('RF labelling is disabled. After embeddings are complete, set '
          'RUN_PREPROCESSING_ALL_FOLDS=True to label d1..d5, or RUN_PREPROCESSING=True for FOLD.')


train raw unique pairs: 183964
test raw unique pairs: 9203
[WARNING] d1/train: 2 proteins excluded because they have no usable ligands.
Protein IDs: ['p69', 'p517']
train 125440 pairs; 1040 usable proteins
test 13836 pairs; 104 usable proteins
Validation protein rows: 52 ligand/protein pool unique ligands: 103277


/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Selected RF: Previous RF (loaded) | features: {'kind': 'morgan', 'n_bits': 2048, 'radius': 2} | weights SHA: 88eb0e097a966100
  RF: 8,192/114,975 unique ligands
All-fold canonical ligands: 154898 rejected: {'zero_bonds': 3}
[BLOCKER] ctddg_eq8 pretraining ligand exclusion not verified: {'status': 'FAILED', 'overlapping_unique_ligands': 67708, 'malformed_or_excluded_pretrain_rows': 13, 'toxic_label_counts': {'0': 828802, '1': 329425}, 'label_classes_ok': True, 'examples': ['c1ccc(CN2CCC(n3cnc4cnc5[nH]ccc5c43)CC2)cc1', 'CNc1ccc2ncnc(Nc3cccc(C)c3)c2c1', 'CN(CC(=O)O)c1c(Br)cccc1Nc1ncnc2ccncc12', 'Cc1c(C(=O)C(N)=O)c2c(OCC(=O)O)cccc2n1Cc1cccc(-c2ccccc2)c1', 'C[C@@H](NC(=O)Nc1cc2[nH]nc(-c3cnn(C4COC4)c3)c2cn1)c1ccccc1', 'COc1cc2c(cc1CN1CCN(C)CC1)Cc1c(-c3ccc(-c4ccc(O)cc4)cc3)n[nH]c1-2', 'C=CCN(CC=C)C(=O)n1cnc(S(=O)(=O)C2CC3CCC2C3)n1', 'COc1ccc(Nc2ccc3c(c2)N(c2nc4c(s2)C(=O)NC(C)(C)C4)CCO3)nn1', 'N#Cc1nccc(N2C[C@@H](S(=O)(=O)c3ccccc3C(F)(F)F)C[C@H]2C(=O)NC2(C#N)CC2)n1', 'COc1cc(-c2cnn(C)c2)ccc1Nc

/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  RF: 8,192/107,086 unique ligands
All-fold canonical ligands: 154898 rejected: {'zero_bonds': 3}
[BLOCKER] ctddg_eq8 pretraining ligand exclusion not verified: {'status': 'FAILED', 'overlapping_unique_ligands': 67708, 'malformed_or_excluded_pretrain_rows': 13, 'toxic_label_counts': {'0': 828802, '1': 329425}, 'label_classes_ok': True, 'examples': ['c1ccc(CN2CCC(n3cnc4cnc5[nH]ccc5c43)CC2)cc1', 'CNc1ccc2ncnc(Nc3cccc(C)c3)c2c1', 'CN(CC(=O)O)c1c(Br)cccc1Nc1ncnc2ccncc12', 'Cc1c(C(=O)C(N)=O)c2c(OCC(=O)O)cccc2n1Cc1cccc(-c2ccccc2)c1', 'C[C@@H](NC(=O)Nc1cc2[nH]nc(-c3cnn(C4COC4)c3)c2cn1)c1ccccc1', 'COc1cc2c(cc1CN1CCN(C)CC1)Cc1c(-c3ccc(-c4ccc(O)cc4)cc3)n[nH]c1-2', 'C=CCN(CC=C)C(=O)n1cnc(S(=O)(=O)C2CC3CCC2C3)n1', 'COc1ccc(Nc2ccc3c(c2)N(c2nc4c(s2)C(=O)NC(C)(C)C4)CCO3)nn1', 'N#Cc1nccc(N2C[C@@H](S(=O)(=O)c3ccccc3C(F)(F)F)C[C@H]2C(=O)NC2(C#N)CC2)n1', 'COc1cc(-c2cnn(C)c2)ccc1Nc1ncc2ccnc(NCC(F)(F)F)c2n1'], 'pretrain_file': '/home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/chembl_final.txt', '

/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  RF: 8,192/115,089 unique ligands
All-fold canonical ligands: 154898 rejected: {'zero_bonds': 3}
[BLOCKER] ctddg_eq8 pretraining ligand exclusion not verified: {'status': 'FAILED', 'overlapping_unique_ligands': 67708, 'malformed_or_excluded_pretrain_rows': 13, 'toxic_label_counts': {'0': 828802, '1': 329425}, 'label_classes_ok': True, 'examples': ['c1ccc(CN2CCC(n3cnc4cnc5[nH]ccc5c43)CC2)cc1', 'CNc1ccc2ncnc(Nc3cccc(C)c3)c2c1', 'CN(CC(=O)O)c1c(Br)cccc1Nc1ncnc2ccncc12', 'Cc1c(C(=O)C(N)=O)c2c(OCC(=O)O)cccc2n1Cc1cccc(-c2ccccc2)c1', 'C[C@@H](NC(=O)Nc1cc2[nH]nc(-c3cnn(C4COC4)c3)c2cn1)c1ccccc1', 'COc1cc2c(cc1CN1CCN(C)CC1)Cc1c(-c3ccc(-c4ccc(O)cc4)cc3)n[nH]c1-2', 'C=CCN(CC=C)C(=O)n1cnc(S(=O)(=O)C2CC3CCC2C3)n1', 'COc1ccc(Nc2ccc3c(c2)N(c2nc4c(s2)C(=O)NC(C)(C)C4)CCO3)nn1', 'N#Cc1nccc(N2C[C@@H](S(=O)(=O)c3ccccc3C(F)(F)F)C[C@H]2C(=O)NC2(C#N)CC2)n1', 'COc1cc(-c2cnn(C)c2)ccc1Nc1ncc2ccnc(NCC(F)(F)F)c2n1'], 'pretrain_file': '/home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/chembl_final.txt', '

/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  RF: 8,192/107,939 unique ligands


## Outputs and run order

1. **Confirm the configuration cell:** repository root, `PROSE_PYTHON` (e.g. the `bioemb` environment's Python), `PROSE_SCRIPT`, `PROSE_SAVED_MODEL`, `PROSE_DEVICE`. Leave `RUN_ALL_FOLD_EMBEDDINGS=True` to process **all five folds**, train and test.
2. **Run from the top on a scheduled compute allocation.** The ProSE cell prints an inventory and checks all ten TXT/FASTA/protein files *before* embedding. It generates missing `.h5` files **directly in their corresponding BindingDB train/test folders**, verifies every output, and reports a final 10/10 summary. Valid existing vectors are reused by default; set `REPLACE_EXISTING_EMBEDDINGS=True` to regenerate them with backups.
3. **RF labelling is separate:** configure `RF_FILE` and `RF_METADATA_FILE`; set `RUN_PREPROCESSING_ALL_FOLDS=True` to create one processed dataset per original fold, or `RUN_PREPROCESSING=True` to create only `FOLD` (default d1). The RF stage is off by default; it does **not** need to be run to generate protein embeddings.
4. Each processed fold is written to a **new** folder `datasets/processed/ctddg_bindingdb_rf_dN_v2/` with `train.tsv`, `val.tsv`, `test.tsv`, `all_bindingdb_ligands.smi`, and `dataset_manifest.json`. Change `PROCESSED_DATASET_VERSION` if an output already exists; previous processed datasets are not overwritten.
5. **Fine-tune per fold** using `02_BindingDB_Finetuning_PyTorch.ipynb`, changing its `DATA_DIR` from the default d1 path to d2, d3, d4, and d5 in turn. Select the matching checkpoint and objective for `ctddg_eq8` or `cdgcn_nll`.

**Data integrity:** no original ligand/protein pairs, unique protein list, or FASTA is rewritten. The `p1..pN` embedding mapping is preserved and checked. The cross-fold ligand leakage audit uses the actual checkpoint's ChEMBL source and is retained; don't treat `FAILED`/`UNVERIFIED` as a pass.
